# Exercises XP - Heart Disease Prediction

In these exercises, I predict if a patient has heart disease. I train Logistic Regression, SVM and XGBoost, each one first with settings I choose myself and then with GridSearchCV, and I compare all the models. I follow the course's pre-filled Colab notebook and complete its TODO parts.

## What you will learn
- Load and inspect CSV data
- EDA and preprocessing
- Train Logistic Regression, SVM, XGBoost
- Hyperparameter tuning with GridSearchCV
- Evaluate with standard metrics

## What you will create
- Working classifiers and a simple comparison report

## How to run in Colab
1. Open [Google Colab](https://colab.research.google.com) → **File → Upload notebook** → choose this file.
2. I keep `Heart Disease Prediction Dataset.zip` from the [course link](https://github.com/devtlv/Datasets-GEN-AI-Bootcamp/raw/refs/heads/main/Week%205/Day%204%20-%20Statistics%20for%20Machine%20Learning/Heart%20Disease%20Prediction%20Dataset.zip) in my `C:\Users\sivan\Datasets` folder.
3. I run the cells from top to bottom. When the upload cell opens a file picker, I choose `Heart Disease Prediction Dataset.zip`. I don't unzip it myself, because the template's Exercise 1 cell extracts it with `zipfile`.

In [ ]:
from google.colab import files
# I import Colab's tool for uploading files from my computer.
uploaded = files.upload()
# I open a file picker and choose "Heart Disease Prediction Dataset.zip".

## Setup

In this cell, I import all the tools I need and I fix the random seed, so I get the same results every time I run the notebook.

In [ ]:
import os, zipfile, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

# Hint: install xgboost in Colab if missing
# !pip install xgboost
# XGBoost is already installed in Colab, so I keep the pip line as a comment.
try:
    from xgboost import XGBClassifier
except Exception:
    XGBClassifier = None

# 42 is the seed: the same number gives the same "random" split and models every run.
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## Exercise 1 - Exploratory Data Analysis

**Instructions**
- Load the data from CSV files
- Remove target column from the training data
- Split the data intro train/test split
- Understand the data

In this exercise, I extract the zip, load the CSV file into a DataFrame and look at its size, columns, types and missing values. I find the target column (`heart disease`), remove it from the features, and split the data into a training set and a testing set.

### What I learn here
- I explore a new dataset before I train any model.
- I keep the target (the answer) separate from the features, so the model cannot see the answer.
- I keep 20% of the rows for testing, with the same share of sick patients in both sets.

In [ ]:
# I extract the dataset zip to an output folder.
ZIP_PATH = 'Heart Disease Prediction Dataset.zip'  # or your path
EXTRACT_DIR = 'heart_ds'
zipfile.ZipFile(ZIP_PATH).extractall(EXTRACT_DIR)
# zipfile opens the zip, and extractall copies the files inside it into the folder heart_ds.

# I list the CSV files under EXTRACT_DIR.
csv_files = glob.glob(os.path.join(EXTRACT_DIR, '**', '*.csv'), recursive=True)
print('CSV files:', csv_files)
csv_path = csv_files[0]  # the zip has only one CSV file, dataset_heart.csv

# I load the CSV into a DataFrame named df.
df = pd.read_csv(csv_path)
# Some column names end with a space (for example 'sex '), so I remove the extra spaces.
df.columns = df.columns.str.strip()

# I inspect df: its size, first rows, column types, missing values and basic statistics.
print(df.shape)
display(df.head())
print(df.dtypes)
print('Missing values in the whole table:', df.isna().sum().sum())
display(df.describe().round(2))

# I identify the target column.
target = 'heart disease'
# In this dataset 1 = no heart disease and 2 = heart disease.
print(df[target].value_counts())
# I change them to 0 and 1, because the metrics and XGBoost expect the classes 0 and 1.
df[target] = df[target].map({1: 0, 2: 1})

# I split features and target.
X = df.drop(columns=[target])  # I remove the target column from the features.
y = df[target]                 # y holds only the answer (0 or 1).

# I do a train test split with stratification.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
# stratify=y keeps the same share of sick patients in the training set and in the testing set.
print('Train:', X_train.shape, '| Test:', X_test.shape)
print('Share of heart disease in train and test:', round(y_train.mean(), 3), round(y_test.mean(), 3))

**What I see:** The printed shape shows **270 patients** and 14 columns (13 features and the target). Every column is a number and there are **0 missing values**. The target `heart disease` has **150 patients without heart disease** (1, now 0) and **120 with heart disease** (2, now 1), so 44.4% are sick and the two classes are close to balanced. After the split, the training set has 216 patients and the testing set 54, and thanks to `stratify=y` both have the same 44.4% share of sick patients. (The column name `serum cholestoral` is spelled like this in the dataset.)

### Basic visual checks

In this cell, I draw histograms of a few number columns to see how their values are spread, and a bar chart of the two classes to check if they are balanced.

In [ ]:
# I pick a few numeric columns and plot histograms.
cols_to_plot = ['age', 'resting blood pressure', 'serum cholestoral', 'max heart rate']
df[cols_to_plot].hist(bins=20, figsize=(10, 6))
plt.suptitle('Histograms of some numeric columns')
plt.tight_layout()
plt.show()

# I plot class balance as a bar chart.
y.value_counts().sort_index().plot(kind='bar')
plt.title('Class balance (0 = no heart disease, 1 = heart disease)')
plt.xlabel('heart disease')
plt.ylabel('Count')
plt.show()

# I compare the mean of each column for the two classes, to see which columns are linked to heart disease.
display(df.groupby(target).mean().T.round(2))

**What I see:** The histograms show that most patients are between 40 and 70 years old, and that cholesterol has a few very high values (up to 564). The bar chart shows 150 healthy and 120 sick patients. The table of means shows the biggest differences between the two groups: patients with heart disease have a lower max heart rate (**138.86** vs **158.33**), more exercise induced angina (0.55 vs 0.15), a higher oldpeak (1.58 vs 0.62), more major vessels (1.15 vs 0.29) and a higher thal code (5.83 vs 3.79). Age, blood pressure and cholesterol differ much less between the two groups.

## Preprocessing pipeline

In this cell, I sort the columns into text columns and number columns, and I build the preprocessing step: `StandardScaler` for the number columns and `OneHotEncoder` for the text columns.

In [ ]:
# I list the object, category or bool columns (text-like columns).
cat_cols = X.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
# I list the numeric columns.
num_cols = X.select_dtypes(include=['number']).columns.tolist()
print('Categorical:', cat_cols)
print('Numeric:', num_cols)

# I build a ColumnTransformer named pre.
pre = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
])
# StandardScaler does (value - training mean) / training standard deviation, so every number column has mean 0 and std 1.
# OneHotEncoder makes one 0/1 column per category. Here cat_cols is empty, because every column is already a number.

**Note:** All 13 feature columns are stored as numbers, so the template's rule gives an empty `cat_cols` and only `StandardScaler` does any work. Some columns, like `chest pain type` or `thal`, are really codes for categories, but I keep the template's rule.

## Helper - evaluation function

In this cell, I complete the template's helper. It predicts the test rows, computes accuracy, precision, recall and F1, draws the confusion matrix, and draws the ROC curve when the model can give probabilities.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

def eval_and_report(name, model, X_test, y_test):
    """Compute metrics and draw confusion matrix and ROC if available.
    Fill in the missing parts.
    """
    # I predict 0 or 1 for every test row.
    y_pred = model.predict(X_test)

    # I compute the metrics dict with accuracy, precision, recall, f1.
    metrics = {
        'accuracy': round(accuracy_score(y_test, y_pred), 4),                     # (TP + TN) / all rows
        'precision': round(precision_score(y_test, y_pred, zero_division=0), 4),  # TP / (TP + FP)
        'recall': round(recall_score(y_test, y_pred, zero_division=0), 4),        # TP / (TP + FN)
        'f1': round(f1_score(y_test, y_pred, zero_division=0), 4),                # 2 * precision * recall / (precision + recall)
    }
    print(name, metrics)

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    # Confusion matrix plot: rows are the true classes, columns are the predicted classes.
    cm = confusion_matrix(y_test, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=['No disease (0)', 'Disease (1)']).plot(ax=axes[0], colorbar=False)
    axes[0].set_title(name + ' - confusion matrix')

    # ROC curve if the model has predict_proba.
    if hasattr(model, 'predict_proba'):
        # Column 1 of predict_proba is the probability of heart disease.
        y_proba = model.predict_proba(X_test)[:, 1]
        fpr, tpr, _ = roc_curve(y_test, y_proba)
        auc = roc_auc_score(y_test, y_proba)
        axes[1].plot(fpr, tpr, label=f'AUC = {auc:.3f}')
        axes[1].plot([0, 1], [0, 1], 'k--', label='Random guess')
        axes[1].set_xlabel('False Positive Rate')
        axes[1].set_ylabel('True Positive Rate')
        axes[1].set_title(name + ' - ROC curve')
        axes[1].legend(loc=4)
        print('AUC:', round(auc, 4))
    else:
        axes[1].axis('off')
        print('This model has no predict_proba, so I skip the ROC curve.')
    plt.tight_layout()
    plt.show()

    return metrics

## Exercise 2 - Logistic Regression without Grid Search

**Instructions**

Use the dataset to build a logistic regression model without using grid search. Split the data into training and testing sets, then train a logistic regression model and evaluate its performance on the test set.

In this exercise, I connect my preprocessing step and a Logistic Regression in one `Pipeline`, with settings I choose myself. I already split the data in Exercise 1, so I train on the training set and evaluate on the test set.

### What I learn here
- I build a baseline model: the starting score I want to improve.
- The Pipeline fits the scaler on the training rows only, so the test rows stay unseen.

In [ ]:
# I create a Pipeline with my preprocessor and LogisticRegression.
# solver='liblinear' works well on small datasets, and max_iter=1000 gives it enough steps to converge.
pipe_lr = Pipeline([
    ('pre', pre),
    ('lr', LogisticRegression(solver='liblinear', max_iter=1000, random_state=RANDOM_STATE)),
])

# I fit on the training data.
pipe_lr.fit(X_train, y_train)

# I evaluate with eval_and_report.
lr_no_gs_metrics = eval_and_report('LR no grid', pipe_lr, X_test, y_test)

**My result:** My baseline Logistic Regression gets **accuracy 0.8519** (46 of the 54 test patients right), precision 0.7857, **recall 0.9167** and **F1 0.8462**, with an AUC of 0.8986. The recall is high: it finds 22 of the 24 sick patients and misses only 2, while 6 healthy patients get a false alarm. For a medical test, missing few sick patients is the more important part.

## Exercise 3 - Logistic Regression with Grid Search

**Instructions**

Build a logistic regression model using the dataset, but this time, use GridSearchCV to optimize the hyperparameters such as C and penalty.

In this exercise, I let GridSearchCV try every combination of `C` and `penalty` with 5-fold cross-validation on the training set, and I evaluate the best model on the test set.

### What I learn here
- `C` controls how strict the model is: a small C means strong regularization (a simpler model), a large C means weak regularization.
- `penalty='l1'` can push some coefficients to exactly 0, `penalty='l2'` only makes them smaller.
- `cv=5` splits the training set into 5 folds, and each fold takes a turn as the validation set.
- `best_score_` is the average validation score, not the test score.

In [ ]:
# I build a Pipeline same as above.
pipe_lr_cv = Pipeline([
    ('pre', pre),
    ('lr', LogisticRegression(solver='liblinear', max_iter=1000, random_state=RANDOM_STATE)),
])

# I define param_grid for lr__C and lr__penalty.
# 'lr__C' means: the parameter C of the Pipeline step named 'lr'.
param_grid = {
    'lr__C': [0.01, 0.1, 1, 10, 100],  # from small to large C values
    'lr__penalty': ['l1', 'l2'],        # 'liblinear' supports 'l1' and 'l2'
}
# 5 values x 2 values = 10 combinations, and with cv=5 that is 50 fits.

# I run GridSearchCV with cv=5 and scoring='f1'.
grid_lr = GridSearchCV(pipe_lr_cv, param_grid, cv=5, scoring='f1', n_jobs=-1)
grid_lr.fit(X_train, y_train)
print('Best params:', grid_lr.best_params_)
print('Best cross-validation F1:', round(grid_lr.best_score_, 4))
# best_estimator_ is the best combination, already refit on the whole training set.
best_lr = grid_lr.best_estimator_
lr_gs_metrics = eval_and_report('LR grid', best_lr, X_test, y_test)

**My result:** The best settings are `C=0.1` and `penalty='l2'`, with an average cross-validation F1 of **0.8146**. On the test set, the tuned model makes exactly the same predictions as my baseline: accuracy 0.8519 and F1 0.8462. Only the AUC is a little higher (**0.9083** instead of 0.8986), so its probabilities rank the patients slightly better. Grid search does not always beat good default settings, especially with only 54 test patients, where one patient changes the accuracy by almost 2%.

## Exercise 4 - SVM without Grid Search

**Instructions**

Train a Support Vector Machine (SVM) classifier on the dataset without using grid search. Choose an appropriate kernel and set the hyperparameters manually.

In this exercise, I train an SVM with settings I choose myself: `kernel='rbf'`, `C=1.0` and `gamma='scale'`.

### What I learn here
- An SVM looks for the boundary that separates the two classes with the widest margin.
- **C = strictness:** how strongly mistakes are penalized.
- **Kernel = shape:** `linear` gives a straight boundary, `rbf` can give a curved one.
- **Gamma = influence:** how far the influence of each training point reaches.

In [ ]:
# I choose the kernel and hyperparameters for SVC.
# kernel='rbf' can learn a curved boundary, C=1.0 is a medium strictness,
# and gamma='scale' calculates gamma from the training data (1 / (number of features * variance)).
pipe_svm = Pipeline([
    ('pre', pre),  # SVM uses distances, so the scaling in pre is important here
    ('svm', SVC(kernel='rbf', C=1.0, gamma='scale', random_state=RANDOM_STATE)),
])
pipe_svm.fit(X_train, y_train)
svm_no_metrics = eval_and_report('SVM no grid', pipe_svm, X_test, y_test)
# I did not set probability=True here, so this model has no predict_proba and no ROC curve.

**My result:** The SVM with my settings gets **accuracy 0.8148**, precision 0.7692, recall 0.8333 and **F1 0.80**. It is a little weaker than Logistic Regression: it finds 20 of the 24 sick patients, so it misses 4 instead of 2.

## Exercise 5 - SVM with Grid Search

**Instructions**

Implement an SVM classifier on the dataset with GridSearchCV to find the best combination of C, kernel, and gamma hyperparameters.

In this exercise, I let GridSearchCV try combinations of `kernel`, `C` and `gamma`, and I evaluate the best SVM on the test set.

### What I learn here
- I search over the three main SVM settings at the same time.
- `probability=True` lets the SVM give probabilities, so I can draw its ROC curve.

In [ ]:
# I build a Pipeline with SVC(probability=True) so I can plot the ROC curve.
pipe_svm_cv = Pipeline([
    ('pre', pre),
    ('svm', SVC(probability=True, random_state=RANDOM_STATE)),
])

# I make a param_grid with svm__kernel, svm__C, svm__gamma.
svm_param_grid = {
    'svm__kernel': ['rbf', 'linear'],
    'svm__C': [0.1, 1, 10, 100],
    'svm__gamma': ['scale', 'auto', 0.01, 0.1],  # gamma only changes the 'rbf' kernel; 'linear' ignores it
}
# 2 x 4 x 4 = 32 combinations, and with cv=5 that is 160 fits.

# I run GridSearchCV, fit it, and evaluate the best estimator.
grid_svm = GridSearchCV(pipe_svm_cv, svm_param_grid, cv=5, scoring='f1', n_jobs=-1)
grid_svm.fit(X_train, y_train)
print('Best params:', grid_svm.best_params_)
print('Best cross-validation F1:', round(grid_svm.best_score_, 4))
best_svm = grid_svm.best_estimator_
svm_gs_metrics = eval_and_report('SVM grid', best_svm, X_test, y_test)

**My result:** The best combination is `kernel='rbf'`, `C=1` and `gamma=0.01`, with an average cross-validation F1 of **0.8246**. The tuned SVM is better than the one from Exercise 4: **accuracy 0.8519** instead of 0.8148, precision 0.8077, recall 0.875 and **F1 0.84** instead of 0.80, with an AUC of 0.9028. The smaller gamma (0.01 instead of `'scale'`, which is 1/13 ≈ 0.077 here) gives every training patient a wider influence, so the boundary is smoother and overfits less.

## Exercise 6 - XGBoost without Grid Search

**Instructions**

Use the dataset to train an XGBoost classifier without hyperparameter tuning. Set the hyperparameters manually and justify your choices.

In this exercise, I train XGBoost with settings I choose myself and I explain each choice.

### What I learn here
- XGBoost builds many small trees one after another, and each new tree corrects the mistakes of the trees before it.
- `learning_rate` is the size of each learning step, `n_estimators` the number of trees, `max_depth` how deep each tree can grow.

**My choices and why:**
- `n_estimators=300`: enough trees to learn the patterns, because each tree only makes a small correction.
- `learning_rate=0.1`: a common middle value. Smaller steps learn more carefully but need more trees.
- `max_depth=3`: the training set has only 216 patients, so I keep each tree shallow to avoid overfitting.
- `subsample=0.8` and `colsample_bytree=0.8`: each tree sees a random 80% of the rows and 80% of the columns, which also helps against overfitting.

In [ ]:
# I build an XGBClassifier and wrap it in a Pipeline with pre.
# Trees don't need scaled numbers, but scaling doesn't hurt them, so I keep the same pre for a fair comparison.
pipe_xgb = Pipeline([
    ('pre', pre),
    ('xgb', XGBClassifier(
        n_estimators=300,       # number of trees
        learning_rate=0.1,      # size of each learning step
        max_depth=3,            # maximum depth of each tree
        subsample=0.8,          # share of the training rows each tree sees
        colsample_bytree=0.8,   # share of the columns each tree sees
        eval_metric='logloss',  # the loss XGBoost reports for a yes/no problem
        random_state=RANDOM_STATE,
    )),
])
pipe_xgb.fit(X_train, y_train)
xgb_no_metrics = eval_and_report('XGB no grid', pipe_xgb, X_test, y_test)

**My result:** XGBoost with my settings gets **accuracy 0.7963**, precision 0.7407, recall 0.8333, **F1 0.7843** and AUC 0.8681. It is the weakest model so far. XGBoost is strongest on big datasets with complex patterns, but here there are only 216 training patients, and the simpler Logistic Regression does better.

## Exercise 7 - XGBoost with Grid Search

**Instructions**

Train an XGBoost classifier on the dataset using GridSearchCV to optimize hyperparameters such as learning_rate, n_estimators, max_depth, etc.

In this exercise, I let GridSearchCV try combinations of the main XGBoost settings, and I evaluate the best model on the test set.

### What I learn here
- I keep the grid small, because the number of fits grows very fast with every value I add.

In [ ]:
# I build a Pipeline with XGBClassifier.
pipe_xgb_cv = Pipeline([
    ('pre', pre),
    ('xgb', XGBClassifier(eval_metric='logloss', random_state=RANDOM_STATE)),
])

# I define a reasonable param_grid.
xgb_param_grid = {
    'xgb__n_estimators': [100, 300],
    'xgb__learning_rate': [0.05, 0.1],
    'xgb__max_depth': [2, 3, 5],
    'xgb__subsample': [0.8, 1.0],
    'xgb__colsample_bytree': [0.8, 1.0],
}
# 2 x 2 x 3 x 2 x 2 = 48 combinations, and with cv=5 that is 240 fits.

# I run GridSearchCV, fit it, and evaluate the best estimator.
grid_xgb = GridSearchCV(pipe_xgb_cv, xgb_param_grid, cv=5, scoring='f1', n_jobs=-1)
grid_xgb.fit(X_train, y_train)
print('Best params:', grid_xgb.best_params_)
print('Best cross-validation F1:', round(grid_xgb.best_score_, 4))
best_xgb = grid_xgb.best_estimator_
xgb_gs_metrics = eval_and_report('XGB grid', best_xgb, X_test, y_test)

**My result:** The best combination is `n_estimators=300`, `learning_rate=0.05`, `max_depth=3`, `subsample=0.8` and `colsample_bytree=1.0`, with an average cross-validation F1 of **0.8062**. On the test set, the predictions are the same as in Exercise 6 (accuracy 0.7963, F1 0.7843), and only the AUC rises a little, from 0.8681 to **0.8736**. So my manual settings were already close to the best ones in the grid.

## Compare models

In this cell, I put the test metrics of my six models in one table and one bar chart, so I can compare them.

In [ ]:
# I build a comparison DataFrame from my metrics dicts.
summary = {}
summary['LR no grid'] = lr_no_gs_metrics
summary['LR grid'] = lr_gs_metrics
summary['SVM no grid'] = svm_no_metrics
summary['SVM grid'] = svm_gs_metrics
summary['XGB no grid'] = xgb_no_metrics
summary['XGB grid'] = xgb_gs_metrics
summary_df = pd.DataFrame.from_dict(summary, orient='index')
display(summary_df)

# I draw the four metrics of every model side by side.
summary_df.plot(kind='bar', figsize=(10, 4))
plt.title('Test metrics of my six models')
plt.ylabel('Score')
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.legend(loc='lower right')
plt.show()
summary

**My comparison report:**
- The best models on the test set are Logistic Regression (with and without grid search) and the tuned SVM, all with **accuracy 0.8519**. Logistic Regression has the best recall (**0.9167**) and the best F1 (**0.8462**), so it misses the fewest sick patients. The tuned SVM has the best precision (0.8077).
- Grid search helped the SVM the most (F1 from 0.80 to 0.84). For Logistic Regression and XGBoost, it gave the same test predictions as my manual settings.
- XGBoost is the weakest here (F1 0.7843). The dataset is small and the patterns are fairly simple, so a linear model is enough.
- My choice is **Logistic Regression**: it has the highest F1 and recall, it is the simplest and fastest model, and its coefficients are easy to explain. With only 54 test patients, one patient changes the accuracy by about 1.9%, so the differences between the top models are small.